# သင်ခန်းစာ ၁၇ - Foundry Local နှင့် Qwen ဖြင့် ဒေသန္တရ AI အေးဂျင့်များ ဖန်တီးခြင်း

ဒီ notebook မှာ သင့်လုပ်ငန်းစက်ပစ္စည်းပေါ်မှာ လုံး၀ပြေးဆွဲနိုင်တဲ့ **ဒေသန္တရ သင်္ချာဆိုင်ရာ အကူအညီပေးသူ** တစ်ယောက်ကို ဖန်တီးပါမယ်။ မည်သည့်အချိန်မှ မိုဃ်းတိမ် inference မသုံးပါ။ အကူအညီပေးသူက:

1. **Foundry Local မှတဆင့် Qwen function calling ဖြင့် တူးလ်များကို ခေါ်နိုင်သည်။**
2. **sandboxed project directory အတွင်းရှိ ဖိုင်များစာရင်းပြုစုခြင်းနှင့် ဖတ်ရှုနိုင်ခြင်း။**
3. **ရိုးရှင်းသော သင်္ချာဆိုင်ရာအတိုင်းအတာများဖြင့် ကုဒ်ကို စုပေါင်းခွဲခြမ်းစိတ်ဖြာနိုင်ခြင်း။**
4. **ဒေသန္တရ RAG (Chroma) ဖြင့် စာရွက်စာတမ်းများ ရှာဖွေနိုင်ခြင်း။**
5. **ဒေသန္တရ MCP ဆာဗာတစ်ခုကို အသုံးပြုနိုင်ခြင်း (မရှိပါက ပျင်းတမ်းတောင်းလျော့ထားမည်။)**

အေးဂျင့်ကုဒ်က မိုဃ်းတိမ်သင်ခန်းစာများနှင့် တကွ 거의နှိုင်းယှဉ်ရလေ့ရှိသော်လည်း — client endpoint က မိုဃ်းတိမ်မှ `localhost` သို့ ပြောင်းလဲထားသည်။


## စတင်ပြင်ဆင်ခြင်း

ဤ notebook ကို run မလုပ်မီ:

1. **Microsoft Foundry Local ကို 설치 설치 설치 (သင့် OS အတွက် [စာရွက်စာတမ်း](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) ကို ကြည့်ပါ။)**
2. **Qwen မော်ဒယ်ကို ဒေါင်းလုပ်လုပ်ပြီး စတင်ပါ:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. အောက်ပါ Python package များကို 설치 설치 설치ပါ။

အားလုံးကို ဒေသတွင်းမှ စတင်လုပ်ဆောင်သည်။ RAM ~8 GB ပါဝင်သော ကြမ်းတမ်းစက်တစ်လုံး သက်ဆိုင်သည့် အနည်းဆုံးဖြစ်သည်။


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Foundry Local နှင့် ချိတ်ဆက်ရန်

`FoundryLocalManager` သည် မော်ဒယ်ကို လိုအပ်ပါက ဒေါင်းလုပ်လုပ်ပြီး ဒေသတွင်းဝန်ဆောင်မှုကို စတင်ပေးကာ **OpenAI-နှင့်ကိုက်ညီသော endpoint** ကို ပေးပါသည်။ ထို့နောက် ကျွန်ုပ်တို့မှာ OpenAI SDK စံနမူနာကို အဲဒီမှာ ဦးတည်စေပါတယ်။ API key သည် ဒေသတွင်း placeholder ဖြစ်ပြီး ကာလောဒ် အတည်ပြုချက် မပါဝင်ပါ။


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. ဒေသခံကိရိယာများ (Sandboxed ဖိုင်လည်ပတ်မှုများ)

ငါတို့ဒစ်စ်ပေါ်မှာ နမူနာကျဆုံးသော စီမံကိန်းတစ်ခုကို ဖန်တီးပြီး၊ ထိုစီမံကိန်းအခြေစိုက်ရာတွင် ကိရိယာများကို သတ်မှတ်သည်။ Sandbox စစ်ဆေးမှုအားလုံးကို ဒေသခံအနေဖြင့်ပါအသုံးချတဲ့အခါမှာပါ အရေးကြီးသည်။ မည်သည့်လမ်းကြောင်းမဆို ဖတ်ယူသော ကိရိယာသည် သင့်အသုံးပြုသူ ခွင့်ပြုချက်များဖြင့် လည်ပတ်ပြီး သင်ထိုကဲ့သို့ ထိတွေ့နိုင်သော မည်သည့်အရာကိုမျှ ထိတွေ့နိုင်သည်။


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Chroma နှင့်တကြ ဒေသခံ RAG

ကျွန်ုပ်တို့သည် စာရွက်လိပ်စာအချို့ကို ဒေသခံ Chroma စုစည်းမှုတစ်ခုအတွင်း သွင်းယူပါသည်။ Chroma သည် စနစ်တွင်းတွင် ပြေးဆွဲပြီး ဒစ်စခ်ပေါ်တွင် vector များကို သိမ်းဆည်းပါသည် - ဆာဗာမရှိ၊ မိုးကောင်းကင်မရှိ။ `search_docs` ကိရိယာသည် မေးခွန်းတစ်ခုအတွက် သက်ဆိုင်စွာသော စာရွက်များကို ရယူပေးသည်။


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. ကိရိယာခေါ်ဆိုမှု Loop

ယခုကျွန်ုပ်တို့သည် OpenAI ကိရိယာ စနစ်ကို အသုံးပြု၍ ကိရိယာများကို မော်ဒယ်နှင့် မှတ်ပုံတင်ပြီး သာမန် ကိရိယာခေါ်ဆိုမှု Loop ကို ပြုလုပ်ပါသည်။ မော်ဒယ်သည် ကိရိယာတစ်ခုကို ဖိတ်ခေါ်ပြီး ကျွန်ုပ်တို့သည် ဒေသတြင်း၌ အောင်မြင်စွာ ကိရိယာကို လည်ပတ်စေကာ၊ ရလဒ်ကို ပြန်ဖြည့်သွင်းပြီး မော်ဒယ်မှ နောက်ဆုံးအဖြေထွက်သည်အထိ ထပ်ခါထပ်ခါ ဆက်လုပ်သွားသည်။ Qwen ၏ ယုံကြည်စိတ်ချရသော function calling သည် ဒီအလုပ်ကို စက်ပစ္စည်းပေါ်တွင် ဖျော်ဖြေပေးသည်။


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## ၅။ ဒေသခံ MCP (ရွေးချယ်ရန်)

MCP သည် cloud ဝန်ဆောင်မှုမဟုတ်သော သယ်ယူပို့ဆောင်မှုတစ်ခုဖြစ်သည် — MCP ဆာဗာတစ်ခုသည် `stdio` အပေါ် ဒေသခံလုပ်ငန်းစဉ်အဖြစ် လည်ပတ်နိုင်သည်။ အောက်တွင်ဖော်ပြထားသည့် ဆိုရှယ်အပိုင်းသည် သင်တွင်တစ်ခု Configure လုပ်ထားလျှင် ဒေသခံ MCP ဆာဗာတစ်ခုထိ ဆက်သွယ်နည်းကို ပြသထားသည် (ဥပမာအားဖြင့် ဖိုင်စနစ်ဆာဗာ)။ `LOCAL_MCP_COMMAND` မသတ်မှတ်ထားလျှင် သေချာစွာ ကျော်လွှားသွားပြီး အဲဒီ notebook သည် အဆုံးထိ ချောမွတ်စွာ လည်ပတ်နေဆဲ ဖြစ်သည်။

လုံခြုံရေး မှတ်ချက် - ဒေသခံ MCP ဆာဗာသည် သင့်အသုံးပြုသူ၏ ခွင့်ပြုချက်များဖြင့် လည်ပတ်သည်။ ၎င်းအား project directory တစ်ခုထဲသို့ အကန့်အသတ်ပေးပါ၊ ထို့နောက် ၎င်းထုတ်လွှတ်ချက်များအား လုပ်ဆောင်မတိုင်မီ စစ်ဆေးပါ။


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## အကျဉ်းချုပ်

သင်သည် သင်၏စက်ပေါ်တွင် လုံးဝလည်ပတ်သော အင်ဂျင်နီယာ အကူအညီပေးရေး အစီအစဉ်တစ်ခုကို တည်ဆောက်ခဲ့သည်။

- **Foundry Local** သည် OpenAI-ကိုက်ညီသည့် အဆုံးမှတ်ချက်နောက်ကွယ်တွင် **Qwen** မော်ဒယ်ကို ပေးအပ်ခဲ့ပြီး ကိုယ်စားလှယ်ကုဒ်သည် မိုးကောင်းကင်သင်ခန်းစာများနှင့် ကိုက်ညီသည်။
- **Sandboxed tools** သည် ပရောဂျက်ဖိုင်ဒိုက်မြက်မှထွက်မလာဘဲ ဖိုင်ဝင်ရာနှင့်ကုဒ်သုံးသပ်ခြင်းကို ကိုယ်စားလှယ်အား ပေးခဲ့သည်။
- **Chroma** သည် ဒေါက်ယူမင့်များမှ **ဒေသဆိုင်ရာ RAG** ပံ့ပိုးပေးခဲ့သည်။
- **Local MCP** သည် MCP ပတ်ဝန်းကျင်ကို အော့ဖ်လိုင်းတွင် အသုံးပြုနည်းကို ပြသခဲ့သည်။

မည်သည့်အချိန်တွင်မဆို မိုးကောင်းကင်အတွင်း inference မဖြစ်ခဲ့ပါ။

### စိန်ခေါ်မှု

ဒေသဆိုင်ရာ ကိုယ်စားလှယ်အား တိုးချဲ့ပါ။

1. **MCP ဆာဗာများအများအပြားနှင့် ချိတ်ဆက်ခြင်း** — ဖိုင်စနစ်ဆာဗာနှင့် git ဆာဗာကို ချိတ်ဆက်ပြီး ကိုယ်စားလှယ်အား အဆိုပါများအကြား ရွေးချယ်ခွင့် ပေးပါ။
2. **ဒေသဆိုင်ရာ မှတ်ဉာဏ် အသုံးပြုခြင်း** — အတိုချုပ် စကားပြောအတိတ်ကာလကို ဒစ်စ်ခ်ပေါ်တွင် သိုလှောင်ထား၍ အကူအညီပေးသူသည် notebook ကို သက်တမ်းအလိုက် ပြန်စလုပ်ရာတွင် အစီအစဉ်များကို မှတ်မှတ်သားသား ရှိစေရန်။
3. **ဒေသခံ အကျုံးဝင် ကိုယ်စားလှယ်များ၏ ပေါင်းစည်းညှိနှိုင်းမှု ပံ့ပိုးခြင်း** — ဒုတိယ ဒေသခံ ကိုယ်စားလှယ်တစ်ယောက် (ဥပမာ အကြောင်းပြသူ) ထည့်သွင်းပြီး သူနှစ်ဦးစလုံး အလုပ်ကို ပူးပေါင်းဆောင်ရွက်စေရန်။

နောက်တစ်ခေါက်သင်ခန်းစာတွင် သင်သည် ထည့်သွင်းထားသည့် AI ကိုယ်စားလှယ်များကို ဘယ်လိုလုံခြုံစေရမည်ကို သင်ယူရမည်ဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
